# Unsloth Gemma 4 GGUF Chat — debug-friendly Colab

Run cells one-by-one. This notebook is intentionally split so you can see exactly whether the slow/failing step is GPU detection, cloning, llama.cpp install, launcher write, or model load.

Recommended runtime: **GPU / T4**. The final launch cell must keep running while you chat.


In [ ]:
# Cell 1 — config and runtime check.
import os
import pathlib
import shutil
import subprocess
import time

os.environ.setdefault('UNSLOTH_CHAT_MODEL', 'unsloth/gemma-4-E4B-it-GGUF:UD-Q4_K_XL')
os.environ.setdefault('UNSLOTH_CHAT_CTX', '4096')
os.environ.setdefault('UNSLOTH_CHAT_PORT', '8888')
# Optional examples:
# os.environ['UNSLOTH_CHAT_EXTRA_ARGS'] = '--temp 0.7 --top-p 0.9'
# os.environ['UNSLOTH_CHAT_GPU'] = 'off'
# os.environ['UNSLOTH_FORCE_LLAMA_INSTALL'] = '1'  # force reinstall/validate llama-server

UNSLOTH_REPO_DIR = pathlib.Path('/content/unsloth')
LLAMA_CPP_DIR = pathlib.Path.home() / '.unsloth' / 'llama.cpp'
LLAMA_SERVER_CANDIDATES = [
    LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-server',
    LLAMA_CPP_DIR / 'llama-server',
]

def run(cmd, **kwargs):
    print('\n$ ' + ' '.join(map(str, cmd)), flush=True)
    started = time.time()
    result = subprocess.run(cmd, **kwargs)
    print(f'→ exit={result.returncode} elapsed={time.time() - started:.1f}s', flush=True)
    return result

def has_gpu_tool():
    return any(shutil.which(name) for name in ('nvidia-smi', 'rocminfo', 'amd-smi', 'hipconfig', 'hipinfo'))

print('Model:', os.environ['UNSLOTH_CHAT_MODEL'])
print('Context:', os.environ['UNSLOTH_CHAT_CTX'])
print('UI port:', os.environ['UNSLOTH_CHAT_PORT'])
print('GPU tool detected:', has_gpu_tool())
if shutil.which('nvidia-smi'):
    run(['nvidia-smi'], check=False)


In [ ]:
# Cell 2 — clone or reuse upstream Unsloth.
if not (UNSLOTH_REPO_DIR / '.git').exists():
    run([
        'git', 'clone', '--depth', '1', '--branch', 'main',
        'https://github.com/unslothai/unsloth.git', str(UNSLOTH_REPO_DIR),
    ], check=True)
else:
    print(f'Using existing checkout: {UNSLOTH_REPO_DIR}')
    run(['git', 'rev-parse', '--short', 'HEAD'], cwd=UNSLOTH_REPO_DIR, check=False)


In [ ]:
# Cell 3 — install or reuse llama-server.
# This is normally the slow step on first run. Reruns skip it if llama-server already exists.
existing = next((p for p in LLAMA_SERVER_CANDIDATES if p.exists()), None)
force_install = os.environ.get('UNSLOTH_FORCE_LLAMA_INSTALL', '0') == '1'
if existing and not force_install:
    print(f'Using existing llama-server: {existing}')
else:
    published_repo = os.environ.get('UNSLOTH_LLAMA_PUBLISHED_REPO')
    if not published_repo:
        published_repo = 'unslothai/llama.cpp' if has_gpu_tool() else 'ggml-org/llama.cpp'
    LLAMA_CPP_DIR.parent.mkdir(parents=True, exist_ok=True)
    install_cmd = [
        'python', 'studio/install_llama_prebuilt.py',
        '--install-dir', str(LLAMA_CPP_DIR),
        '--llama-tag', os.environ.get('UNSLOTH_LLAMA_TAG', 'latest'),
        '--published-repo', published_repo,
        '--simple-policy',
    ]
    print('Installing llama.cpp prebuilt via Unsloth installer:', published_repo)
    result = run(install_cmd, cwd=UNSLOTH_REPO_DIR, check=False)
    if result.returncode != 0 and published_repo != 'ggml-org/llama.cpp':
        print('Primary prebuilt repo failed; retrying ggml-org/llama.cpp...')
        install_cmd[install_cmd.index('--published-repo') + 1] = 'ggml-org/llama.cpp'
        result = run(install_cmd, cwd=UNSLOTH_REPO_DIR, check=False)
    if result.returncode != 0:
        raise RuntimeError('llama.cpp prebuilt install failed. The real installer log is above this traceback.')
    existing = next((p for p in LLAMA_SERVER_CANDIDATES if p.exists()), None)
    print('Installed llama-server:', existing)


In [ ]:
# Cell 4 — write the custom chat launcher into /content.
import base64
LAUNCHER_B64 = (
    "..."
)
launcher = base64.b64decode(LAUNCHER_B64).decode('utf-8')
launcher_path = pathlib.Path('/content/unsloth_gemma_chat_launch.py')
launcher_path.write_text(launcher, encoding='utf-8')
print('Wrote launcher:', launcher_path)


In [ ]:
# Cell 5 — launch chat UI. Keep this cell running while chatting.
run(['python', str(launcher_path)], check=True)
